# 07 - Model Comparison: ImageNet vs VGGFace2

Answers Q3: **does face-specific training buy you FFA prediction, over and
above generic object training?**

Both encoders are ResNet50. Same depth, same width, same 2048-d feature
count, same encoding model, same folds. The *only* thing that differs is
the training diet (ImageNet objects vs VGGFace2 faces) and the
preprocessing each set of weights requires. That is what makes the
contrast interpretable.

In [7]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

c:\Users\user\Documents\GitHub\ffa-dnn-ablation\notebooks\ffa-dnn-ablation\ffa-dnn-ablation\ffa-dnn-ablation


Cloning into 'ffa-dnn-ablation'...


In [ ]:
!pip install nilearn decord python-dotenv gdown --quiet

## Load dataset link from Colab Secrets

Requires a secret named `DROPBOX_DATASET_LINK` set via the key icon in
the left sidebar, with notebook access turned on.

In [9]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")
print("Link loaded:", os.environ["DROPBOX_DATASET_LINK"] is not None)

ModuleNotFoundError: No module named 'google'

In [ ]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.models.resnet_vggface2_wrapper import (
    get_resnet50_encoder,
    get_vggface2_encoder,
    assert_not_imagenet,
)
from src.encoding_model import train_and_evaluate_encoding_model, region_mean_accuracy

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
download_algonauts_data(data_dir="data/raw")

## Load fMRI ROI data

Regions are ordered early visual -> higher-level throughout this notebook,
so any hierarchy or selectivity pattern is easy to read off the x-axis.

In [ ]:
fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"

roi_data = load_all_rois(fmri_dir, subject)

for roi_name, data in roi_data.items():
    print(f"{roi_name}: {data.shape}")

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]
print(f"\nNumber of training videos (from fMRI data): {NUM_TRAIN_VIDEOS}")

REGION_ORDER = ["V1", "V2", "V3", "V4", "LOC", "EBA", "FFA", "STS", "PPA"]

## Load video list

The video folder contains 1102 files: 1000 training videos with fMRI
labels, plus 102 held-out test videos from the Algonauts challenge that
have no fMRI data here. Keep only the first `NUM_TRAIN_VIDEOS`, matched to
the fMRI data itself rather than hardcoded.

In [ ]:
video_dir = "data/raw/AlgonautsVideos268_All_30fpsmax"
video_paths = get_video_list(video_dir)

print(f"Total video files found: {len(video_paths)}")

video_paths = video_paths[:NUM_TRAIN_VIDEOS]

assert len(video_paths) == NUM_TRAIN_VIDEOS, (
    f"Video count ({len(video_paths)}) does not match fMRI row count "
    f"({NUM_TRAIN_VIDEOS}). Check the video folder contents before continuing."
)

print(f"Using {len(video_paths)} training videos (matched to fMRI data)")

## Feature extraction

One helper, used for both models. The model and its transform are passed
in together and never mixed: ImageNet weights get the ImageNet pipeline
(RGB, [0,1], mean/std normalize), VGGFace2 weights get the Caffe pipeline
(BGR, [0,255], mean subtraction only).

Preprocessing is the one thing that must **not** be matched across the two
models. Feeding Caffe-converted face weights ImageNet-normalized input
would put them out of distribution, and the resulting failure would read
as "face features do not predict FFA" when it is really "the input was
wrong."

Features are cached per model, and the cache is invalidated on video-count
mismatch rather than silently reused.

In [ ]:
def extract_features(model, transform, video_paths, cache_path, num_frames=8):
    """
    Extracts one pooled feature vector per video: num_frames frames are
    passed through the encoder and averaged.

    Returns
    -------
    np.ndarray, shape (num_videos, 2048)
    """
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    if os.path.exists(cache_path):
        cached = np.load(cache_path)
        if cached.shape[0] == len(video_paths):
            print(f"Cache matches ({cached.shape[0]} videos). Using {cache_path}")
            return cached
        print(
            f"Cache has {cached.shape[0]} videos but current list has "
            f"{len(video_paths)}. Cache is stale, re-extracting."
        )

    print(f"Extracting features -> {cache_path} (this will take a while)...")
    feats_all = []
    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=num_frames)
        batch = torch.stack([transform(Image.fromarray(f)) for f in frames]).to(device)
        with torch.no_grad():
            feats = model(batch)                # (num_frames, 2048)
        feats_all.append(feats.mean(dim=0).cpu().numpy())

    X = np.stack(feats_all, axis=0)
    np.save(cache_path, X)
    print("Saved features to", cache_path)
    return X

### ImageNet-trained ResNet50

In [ ]:
imagenet_model, imagenet_transform = get_resnet50_encoder(device=device)

X_imagenet = extract_features(
    imagenet_model, imagenet_transform, video_paths,
    cache_path="data/processed/resnet50_imagenet_features.npy",
)
print("ImageNet feature shape:", X_imagenet.shape)

### VGGFace2-trained ResNet50

`arch="resnet50_scratch"` is trained from scratch on VGGFace2, so its diet
is purely faces. (`resnet50_ft` was pre-trained on MS1M first, which muddies
the contrast.)

`assert_not_imagenet` compares actual conv1 tensors against both torchvision
ImageNet checkpoints, so it cannot be fooled by a function that merely claims
to have loaded face weights. If this raises, stop: every number downstream
would be meaningless.

In [ ]:
vgg_model, vgg_transform = get_vggface2_encoder(device=device, arch="resnet50_scratch")

# Hard stop if the face encoder is secretly an ImageNet encoder.
assert_not_imagenet(vgg_model)

X_vggface2 = extract_features(
    vgg_model, vgg_transform, video_paths,
    cache_path="data/processed/resnet50_vggface2_features.npy",
)
print("VGGFace2 feature shape:", X_vggface2.shape)

### Sanity checks before any regression

Catches a shape mismatch here, rather than three cells later with a cryptic
IndexError. The final check confirms the two feature matrices are actually
different numbers: identical features would mean both encoders carry the
same weights.

In [ ]:
for name, X in [("ImageNet", X_imagenet), ("VGGFace2", X_vggface2)]:
    assert X.shape[0] == NUM_TRAIN_VIDEOS, (
        f"{name} features have {X.shape[0]} rows but fMRI data has "
        f"{NUM_TRAIN_VIDEOS} rows. These must match."
    )

for roi_name, data in roi_data.items():
    assert data.shape[0] == NUM_TRAIN_VIDEOS, (
        f"ROI {roi_name} has {data.shape[0]} rows, expected {NUM_TRAIN_VIDEOS}."
    )

assert not np.allclose(X_imagenet, X_vggface2), (
    "ImageNet and VGGFace2 feature matrices are identical. Both encoders "
    "are carrying the same weights -- the contrast would be meaningless."
)

print("All shapes match, and the two encoders produce different features. "
      "Safe to proceed.")

## Per-region encoding accuracy for both models

Note on alpha: `train_and_evaluate_encoding_model` uses `RidgeCV` with
`alpha_per_target=True` and selects the regularization strength per voxel
via internal cross-validation. There is deliberately no hand-picked alpha
here -- notebook 04's validation showed a fixed alpha was not well
justified (the fc7 vs fc8 ranking flipped depending on the value chosen).

Both models go through the identical function, folds, seed, and alpha
search, so any difference in the results is attributable to the training
diet and nothing else.

In [ ]:
def score_all_regions(X, roi_data, layer_label, region_order):
    """Runs the encoding model for every ROI and returns a tidy DataFrame."""
    rows = []
    for region_name in region_order:
        Y = roi_data[region_name]
        voxel_scores = train_and_evaluate_encoding_model(X, Y)
        mean_acc = region_mean_accuracy(voxel_scores)
        rows.append({"layer": layer_label, "region": region_name,
                     "accuracy": mean_acc})
        print(f"{layer_label} -> {region_name}: {mean_acc:.4f}")
    return pd.DataFrame(rows)

In [ ]:
os.makedirs("results/tables/resnet50", exist_ok=True)
os.makedirs("results/tables/vggface2", exist_ok=True)

print("=== ImageNet ResNet50 ===")
img_df = score_all_regions(X_imagenet, roi_data, "resnet_final", REGION_ORDER)
img_df.to_csv("results/tables/resnet50/final_layer_accuracy.csv", index=False)

print("\n=== VGGFace2 ResNet50 ===")
vgg_df = score_all_regions(X_vggface2, roi_data, "vggface2_final", REGION_ORDER)
vgg_df.to_csv("results/tables/vggface2/final_layer_accuracy.csv", index=False)

## Build the comparison table

In [ ]:
comp = (
    img_df[["region", "accuracy"]].rename(columns={"accuracy": "imagenet"})
    .merge(
        vgg_df[["region", "accuracy"]].rename(columns={"accuracy": "vggface2"}),
        on="region",
    )
    .set_index("region")
    .reindex(REGION_ORDER)
)
comp["difference"] = comp["vggface2"] - comp["imagenet"]

# Second net behind assert_not_imagenet: identical accuracies in every
# region means the two encoders produced the same features.
if np.allclose(comp["imagenet"], comp["vggface2"]):
    raise RuntimeError(
        "ImageNet and VGGFace2 accuracies are identical across every region. "
        "The face-vs-ImageNet contrast is really ImageNet-vs-ImageNet. "
        "Re-run assert_not_imagenet() and check the weight loading."
    )

comp.round(4)

## The figure

Top panel: raw accuracy per region, two diets side by side.

Bottom panel: the difference. **This panel is the experiment.** If face
training is selectively useful, the positive bars should concentrate in the
face-responsive regions (FFA, and plausibly EBA/STS), while V1-V4 sit near
zero. Uniformly positive bars everywhere would instead say "VGGFace2 is
just a better general encoder here" -- a different claim, and not a
face-selectivity result.

In [ ]:
x = np.arange(len(REGION_ORDER))
w = 0.4
ffa_i = REGION_ORDER.index("FFA")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

ax1.bar(x - w / 2, comp["imagenet"], w, label="ResNet50 / ImageNet", color="#9aa7b3")
ax1.bar(x + w / 2, comp["vggface2"], w, label="ResNet50 / VGGFace2", color="#e07a5f")
ax1.set_ylabel("Mean correlation (accuracy)")
ax1.set_title("Encoding accuracy by region: ImageNet vs VGGFace2 (ResNet50)")
ax1.legend()
ax1.axvspan(ffa_i - 0.5, ffa_i + 0.5, color="gold", alpha=0.15, zorder=0)

diff = comp["difference"].values
ax2.bar(x, diff, color=["#3d9970" if d >= 0 else "#b04a4a" for d in diff])
ax2.axhline(0, color="black", lw=0.8)
ax2.axvspan(ffa_i - 0.5, ffa_i + 0.5, color="gold", alpha=0.15, zorder=0)
ax2.set_ylabel("VGGFace2 - ImageNet")
ax2.set_title("Face-training advantage per region (positive = VGGFace2 predicts better)")
ax2.set_xticks(x)
ax2.set_xticklabels(REGION_ORDER, rotation=45, ha="right")
ax2.set_xlabel("Brain region (early visual -> higher-level)")

fig.tight_layout()
os.makedirs("results/figures/vggface2", exist_ok=True)
fig.savefig("results/figures/vggface2/imagenet_vs_vggface2.png", dpi=150)
plt.show()

## Summary: is FFA where face training helps most?

The clearest single summary of the result, and a caption-ready number.

In [ ]:
comp["gain_rank"] = comp["difference"].rank(ascending=False)

ffa_gain = comp.loc["FFA", "difference"]
ffa_rank = int(comp.loc["FFA", "gain_rank"])

print(f"FFA gain from face training: {ffa_gain:+.4f}")
print(f"FFA rank among {len(REGION_ORDER)} regions by gain: "
      f"{ffa_rank} (1 = largest gain)")
print(f"\nMean gain in early visual (V1-V4): "
      f"{comp.loc[['V1','V2','V3','V4'], 'difference'].mean():+.4f}")
print(f"Mean gain in face-responsive (FFA, STS): "
      f"{comp.loc[['FFA','STS'], 'difference'].mean():+.4f}")

print("\nRanked by face-training advantage:")
print(comp.sort_values("difference", ascending=False)[["difference"]].round(4))

comp.to_csv("results/tables/vggface2/imagenet_vs_vggface2.csv")

### Interpreting this

- **FFA rank 1, early visual near zero** -> supports face-specific training
  producing FFA-like representations. This is the hypothesis.
- **FFA positive but so is everything else** -> VGGFace2 is a better encoder
  overall in this dataset; not a selectivity result.
- **FFA near zero** -> a null. Before believing it, confirm
  `assert_not_imagenet` passed and that the Caffe preprocessing constants in
  `resnet_vggface2_wrapper.py` were verified against
  `datasets/vgg_face2.py` in `cydonia999/VGGFace2-pytorch`. A preprocessing
  error produces a null that looks exactly like a real one.

These are descriptive differences with no error bars. To claim a *reliable*
FFA advantage, the difference needs a significance test -- a permutation
test over the difference score, or bootstrapping across voxels, in
`src/stats.py`.

In [ ]:
from google.colab import files

files.download("results/figures/vggface2/imagenet_vs_vggface2.png")
files.download("results/tables/vggface2/imagenet_vs_vggface2.csv")
files.download("results/tables/vggface2/final_layer_accuracy.csv")